# Disease Ortholog TF Expression Filter

Load disease orthologs, known TFs, and the expression summary; then build a final disease-TF gene list expressed above configurable cutoffs in at least `N` broad cell types.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = Path("/Users/nick/Projects/data/morphseq/results/20260612")
OUT_DIR = Path("/Users/nick/Projects/repositories/morphseq/results/nlammers/20260612")

DISEASE_PATH = DATA_DIR / "gene2DiseaseViaOrthology_2026.05.21.csv"
EXPRESSION_PATH = DATA_DIR / "gene_expression_summary.csv"
TF_PATH = DATA_DIR / "zscapetools_tfs.csv"

# Common-sense defaults. Tune these and re-run from this cell downward.
MIN_CELL_TYPES = 3
MIN_AVG_EXPR = 1.0
MIN_FRAC_EXPR = 0.25
CELL_TYPE_COL = "cell_type_broad"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 20)

In [2]:
disease = pd.read_csv(DISEASE_PATH, skiprows=1, encoding="utf-8-sig")

# The current expression CSV has a truncated final line, so use the Python engine
# and skip malformed records. This retains all complete rows.
expression = pd.read_csv(EXPRESSION_PATH, engine="python", on_bad_lines="skip")

tfs = pd.read_csv(TF_PATH)

print(f"Disease ortholog table: {disease.shape[0]:,} rows x {disease.shape[1]:,} columns")
print(f"Expression summary: {expression.shape[0]:,} rows x {expression.shape[1]:,} columns")
print(f"Known TF table: {tfs.shape[0]:,} rows x {tfs.shape[1]:,} columns")

display(disease.head(3))
display(expression.head(3))
display(tfs.head())

Disease ortholog table: 9,090 rows x 10 columns
Expression summary: 10,939,814 rows x 6 columns
Known TF table: 2,547 rows x 5 columns


,Zebrafish Gene ID,Zebrafish Gene Symbol,Human Ortholog Entrez Gene Id,Human Ortholog Symbol,DO Term Name,DO Term ID,OMIM Term Name,OMIM ID,Evidence Code,Publication
0,ZDB-GENE-030912-4,aaas,8086,AAAS,triple-A syndrome,DOID:0050602,Achalasia-addisonianism-alacrimia syndrome,231550.0,ECO:0000266,ZDB-PUB-170210-12
1,ZDB-GENE-040718-120,aagab,79719,AAGAB,punctate palmoplantar keratoderma type I,DOID:0080214,"Keratoderma, palmoplantar, punctate type IA",148600.0,ECO:0000266,ZDB-PUB-170210-12
2,ZDB-GENE-030131-3663,aars1,16,AARS1,NaN,NaN,"?Leukoencephalopathy, hereditary diffuse, with...",619661.0,ECO:0000266,ZDB-PUB-170210-12


,cell_type_broad,timepoint,n_cells,gene,avg_expr,frac_expr
0,HR ionocyte,12,10,ENSDARG00000000001,0.0,0.0
1,HR ionocyte,12,10,ENSDARG00000000002,0.0,0.0
2,HR ionocyte,12,10,ENSDARG00000000018,0.0,0.0


,id,gene_short_name,family,protein,entrez_id
0,ENSDARG00000112884,BX510909.2,zf-C2H2,ENSDARP00000153329;,-
1,ENSDARG00000039095,nkx2,Homeobox,ENSDARP00000123711;ENSDARP00000057093;,30698
2,ENSDARG00000102994,znf1034,zf-C2H2,ENSDARP00000139270;,-
3,ENSDARG00000088084,si:dkey-66i24,zf-C2H2,ENSDARP00000108335;,570015
4,ENSDARG00000094242,znf1171,zf-C2H2,ENSDARP00000119036;,-


In [3]:
def clean_symbol(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower()


def add_ab_paralog_grouping(genes: pd.DataFrame) -> tuple[pd.DataFrame, set[str]]:
    genes = genes.copy()
    genes["paralog_suffix"] = genes["zf_symbol_key"].str.extract(r"([ab])$", expand=False)
    genes["paralog_stem"] = genes["zf_symbol_key"].str.replace(r"([ab])$", "", regex=True)

    suffix_counts = (
        genes.loc[genes["paralog_suffix"].notna()]
        .groupby("paralog_stem")["paralog_suffix"]
        .nunique()
    )
    ab_paralog_stems = set(suffix_counts[suffix_counts >= 2].index)

    genes["effective_gene_key"] = genes["zf_symbol_key"]
    genes["is_ab_paralog_group"] = genes["paralog_stem"].isin(ab_paralog_stems)
    genes.loc[genes["is_ab_paralog_group"], "effective_gene_key"] = genes.loc[
        genes["is_ab_paralog_group"], "paralog_stem"
    ]

    return genes, ab_paralog_stems


disease = disease.copy()
tfs = tfs.copy()

disease["zf_symbol_key"] = clean_symbol(disease["Zebrafish Gene Symbol"])
tfs["tf_symbol_key"] = clean_symbol(tfs["gene_short_name"])

disease_genes = (
    disease[["Zebrafish Gene ID", "Zebrafish Gene Symbol", "zf_symbol_key"]]
    .drop_duplicates()
    .sort_values(["Zebrafish Gene Symbol", "Zebrafish Gene ID"])
)
disease_genes, ab_paralog_stems = add_ab_paralog_grouping(disease_genes)

disease = disease.merge(
    disease_genes[
        [
            "Zebrafish Gene ID",
            "zf_symbol_key",
            "paralog_suffix",
            "paralog_stem",
            "effective_gene_key",
            "is_ab_paralog_group",
        ]
    ],
    on=["Zebrafish Gene ID", "zf_symbol_key"],
    how="left",
)

print(f"Total unique zebrafish disease ortholog genes: {disease_genes.shape[0]:,}")
print(f"Unique disease ortholog gene symbols: {disease_genes['zf_symbol_key'].nunique():,}")
print(
    "Effective disease genes after collapsing represented a/b paralog groups: "
    f"{disease_genes['effective_gene_key'].nunique():,}"
)
print(f"Represented a/b paralog groups collapsed: {len(ab_paralog_stems):,}")

Total unique zebrafish disease ortholog genes: 5,759
Unique disease ortholog gene symbols: 5,759
Effective disease genes after collapsing represented a/b paralog groups: 4,646
Represented a/b paralog groups collapsed: 1,109


In [4]:
tf_lookup = tfs.rename(
    columns={
        "id": "tf_ensembl_id",
        "gene_short_name": "tf_gene_short_name",
        "family": "tf_family",
        "protein": "tf_protein",
        "entrez_id": "tf_entrez_id",
    }
)

disease_tf = disease.merge(
    tf_lookup,
    left_on="zf_symbol_key",
    right_on="tf_symbol_key",
    how="inner",
)

disease_tf_genes = (
    disease_tf[
        [
            "Zebrafish Gene ID",
            "Zebrafish Gene Symbol",
            "Human Ortholog Entrez Gene Id",
            "Human Ortholog Symbol",
            "tf_ensembl_id",
            "tf_gene_short_name",
            "tf_family",
            "tf_entrez_id",
            "zf_symbol_key",
            "paralog_suffix",
            "paralog_stem",
            "effective_gene_key",
            "is_ab_paralog_group",
        ]
    ]
    .drop_duplicates()
    .sort_values(["Zebrafish Gene Symbol", "tf_ensembl_id"])
)

print(f"Disease ortholog table rows after TF filter: {disease_tf.shape[0]:,}")
print(f"Unique disease TF genes by zebrafish symbol: {disease_tf['zf_symbol_key'].nunique():,}")
print(f"Unique disease TF Ensembl IDs: {disease_tf['tf_ensembl_id'].nunique():,}")
print(
    "Effective disease TF candidates after paralog collapse: "
    f"{disease_tf_genes['effective_gene_key'].nunique():,}"
)

display(disease_tf_genes.head(5))

Disease ortholog table rows after TF filter: 876
Unique disease TF genes by zebrafish symbol: 484
Unique disease TF Ensembl IDs: 516
Effective disease TF candidates after paralog collapse: 363


,Zebrafish Gene ID,Zebrafish Gene Symbol,Human Ortholog Entrez Gene Id,Human Ortholog Symbol,tf_ensembl_id,tf_gene_short_name,tf_family,tf_entrez_id,zf_symbol_key,paralog_suffix,paralog_stem,effective_gene_key,is_ab_paralog_group
0,ZDB-GENE-061215-112,adnpa,23394,ADNP,ENSDARG00000062002,adnpa,Homeobox,564637,adnpa,a,adnp,adnp,True
1,ZDB-GENE-030131-6385,adnpb,23394,ADNP,ENSDARG00000074293,adnpb,Homeobox,100330315,adnpb,b,adnp,adnp,True
2,ZDB-GENE-110411-190,aff2,2334,AFF2,ENSDARG00000052242,aff2,AF-4,100330019,aff2,<NA>,aff2,aff2,False
3,ZDB-GENE-030219-193,aff3,3899,AFF3,ENSDARG00000079575,aff3,AF-4,570215,aff3,<NA>,aff3,aff3,False
4,ZDB-GENE-100910-5,aff4,27125,AFF4,ENSDARG00000001857,aff4,AF-4,556421,aff4,<NA>,aff4,aff4,False


In [5]:
passing_expression = expression.loc[
    (expression["avg_expr"] >= MIN_AVG_EXPR)
    & (expression["frac_expr"] >= MIN_FRAC_EXPR),
    ["gene", CELL_TYPE_COL, "avg_expr", "frac_expr"],
]

passing_cell_type_counts = (
    passing_expression[["gene", CELL_TYPE_COL]]
    .drop_duplicates()
    .groupby("gene", as_index=False)
    .agg(n_passing_cell_types=(CELL_TYPE_COL, "nunique"))
)

expression_gene_stats = (
    expression.groupby("gene", as_index=False)
    .agg(
        max_avg_expr=("avg_expr", "max"),
        max_frac_expr=("frac_expr", "max"),
        n_observed_cell_types=(CELL_TYPE_COL, "nunique"),
    )
)

disease_tf_expression = (
    disease_tf_genes.merge(
        passing_cell_type_counts,
        left_on="tf_ensembl_id",
        right_on="gene",
        how="left",
    )
    .merge(
        expression_gene_stats,
        left_on="tf_ensembl_id",
        right_on="gene",
        how="left",
        suffixes=("", "_expr"),
    )
    .drop(columns=["gene", "gene_expr"])
)

disease_tf_expression["n_passing_cell_types"] = (
    disease_tf_expression["n_passing_cell_types"].fillna(0).astype(int)
)
MIN_CELL_TYPES=3
final_genes = (
    disease_tf_expression.query("n_passing_cell_types >= @MIN_CELL_TYPES")
    .sort_values(
        ["n_passing_cell_types", "max_avg_expr", "max_frac_expr", "Zebrafish Gene Symbol"],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)
final_effective_candidates = (
    final_genes.drop_duplicates("effective_gene_key", keep="first")
    .reset_index(drop=True)
)

print(
    "Final disease TF genes passing expression cutoffs "
    f"(N >= {MIN_CELL_TYPES} cell types, avg_expr >= {MIN_AVG_EXPR}, "
    f"frac_expr >= {MIN_FRAC_EXPR}): {final_genes.shape[0]:,}"
)
print(f"Unique final zebrafish symbols: {final_genes['zf_symbol_key'].nunique():,}")
print(f"Unique final Ensembl IDs: {final_genes['tf_ensembl_id'].nunique():,}")
print(
    "Effective final disease TF candidates after paralog collapse: "
    f"{final_effective_candidates.shape[0]:,}"
)

display(final_effective_candidates.head(5))

Final disease TF genes passing expression cutoffs (N >= 3 cell types, avg_expr >= 1.0, frac_expr >= 0.25): 68
Unique final zebrafish symbols: 67
Unique final Ensembl IDs: 68
Effective final disease TF candidates after paralog collapse: 56


,Zebrafish Gene ID,Zebrafish Gene Symbol,Human Ortholog Entrez Gene Id,Human Ortholog Symbol,tf_ensembl_id,tf_gene_short_name,tf_family,tf_entrez_id,zf_symbol_key,paralog_suffix,paralog_stem,effective_gene_key,is_ab_paralog_group,n_passing_cell_types,max_avg_expr,max_frac_expr,n_observed_cell_types
0,ZDB-GENE-030131-1989,zbtb16a,7704,ZBTB16,ENSDARG00000007184,zbtb16a,ZBTB,323269,zbtb16a,a,zbtb16,zbtb16,True,64,12.867809,0.974684,114.0
1,ZDB-GENE-060503-246,jarid2b,3720,JARID2,ENSDARG00000062268,jarid2b,ARID,558456,jarid2b,b,jarid2,jarid2,True,58,4.449129,0.893805,114.0
2,ZDB-GENE-030131-5868,cux1a,1523,CUX1,ENSDARG00000078459,cux1a,CUT,797445,cux1a,a,cux1,cux1,True,47,3.017544,0.880952,114.0
3,ZDB-GENE-041203-1,foxp1b,27086,FOXP1,ENSDARG00000014181,foxp1b,Fork_head,569047,foxp1b,b,foxp1,foxp1,True,46,6.194690,0.946903,114.0
4,ZDB-GENE-060613-1,thrab,7067,THRA,ENSDARG00000052654,thrab,THR-like,558427,thrab,b,thra,thra,True,46,3.916667,0.793296,114.0


In [6]:
cutoff_tag = (
    f"N{MIN_CELL_TYPES}_avg{MIN_AVG_EXPR:g}_frac{MIN_FRAC_EXPR:g}"
    .replace(".", "p")
)
output_path = OUT_DIR / f"disease_tf_expression_filtered_{cutoff_tag}.csv"
effective_output_path = OUT_DIR / f"disease_tf_expression_filtered_effective_{cutoff_tag}.csv"

final_genes.to_csv(output_path, index=False)
final_effective_candidates.to_csv(effective_output_path, index=False)
print(f"Saved final gene list to: {output_path}")
print(f"Saved paralog-collapsed final candidate list to: {effective_output_path}")

Saved final gene list to: /Users/nick/Projects/repositories/morphseq/results/nlammers/20260612/disease_tf_expression_filtered_N1_avg1_frac0p25.csv
Saved paralog-collapsed final candidate list to: /Users/nick/Projects/repositories/morphseq/results/nlammers/20260612/disease_tf_expression_filtered_effective_N1_avg1_frac0p25.csv
